# Extract the 12 STEK text files into the `scraped_data_topics` format

Reads the pre-extracted plain-text versions of the 12 STEK documents from
`Input text files/`, chunks them into ~250-word windows, and tags each chunk
against the same 5 LDA topics (k=5, `stek_results/lda_topics_k5.txt`) used to
tag `scraped_data_topics/manifest.json`.

**Output:** `input_data_topics/text/*.txt` (one clean, topic-tagged chunk per
file, same header style as the scraped pages) + `input_data_topics/manifest.json`
(one row per chunk, same schema as `scraped_data_topics/manifest.json`).

Word-window chunking (not page-based) matches `chunk_document()` in the
original `stek_pipeline.py` — i.e. the exact unit the LDA topics were trained
on. The `.txt` files also lack reliable page markers (some have `SEITE N VON N`,
several have none), so a word-window is the only consistent unit across all 12.

This does **not** run embeddings — it produces the second source (alongside
`scraped_data_topics`) to combine into one manifest for a single embedding pass.

No external dependencies: reads `.txt` directly, no PyMuPDF.


In [ ]:
import re
import json
import hashlib
import unicodedata
from pathlib import Path
from collections import Counter


## Config

In [ ]:
TEXT_INPUT_DIR = Path("Input text files")  # pre-extracted text of the 12 STEK docs
LDA_TOPICS_PATH = Path("STEK2035-chatbot/pdfs/stek_output/texts/stek_results/lda_topics_k5.txt")

OUTPUT_DIR = Path("input_data_topics")
TEXT_DIR = OUTPUT_DIR / "text"
MANIFEST_PATH = OUTPUT_DIR / "manifest.json"
TEXT_DIR.mkdir(parents=True, exist_ok=True)

# Matches chunking.target_tokens / min_tokens in config_v2.yaml, the config
# that produced the final k=5 LDA topics -- keeps chunk granularity
# consistent with what the topic model was trained on.
CHUNK_TARGET_TOKENS = 250
CHUNK_MIN_TOKENS = 80


## Step 1 — load the 5 LDA topic keyword lists

Parses gensim's `Topic  N: word, word, ...` output so the *same* 5 topics
(and the same keywords) used to tag `scraped_data_topics` are used here.


In [ ]:
def parse_lda_topics(path: Path) -> dict[int, list[str]]:
    """Parse `Topic  N: word, word, ...` lines into {topic_id: [keywords]}."""
    topics = {}
    pattern = re.compile(r"Topic\s+(\d+):\s*(.+)")
    for line in path.read_text(encoding="utf-8").splitlines():
        m = pattern.match(line.strip())
        if not m:
            continue
        topic_id = int(m.group(1))
        keywords = [w.strip() for w in m.group(2).split(",") if w.strip()]
        topics[topic_id] = keywords
    return topics


TOPIC_KEYWORDS = parse_lda_topics(LDA_TOPICS_PATH)
for tid, kws in sorted(TOPIC_KEYWORDS.items()):
    print(f"Topic {tid}: {', '.join(kws)}")


## Step 2 — cleaning

Same boilerplate-removal patterns as `config_v2.yaml` (the config used for
the final LDA run), so the text a chunk is tagged on matches what the topic
model was actually trained on. Applied to the full document text. Everything
is read and written as UTF-8, avoiding the encoding corruption visible in
`scraped_data_topics` (e.g. `grA?n` instead of `grün`).


In [ ]:
INLINE_PATTERNS = [
    r"(?i)seite\s+\d+\s+von\s+\d*",
    r"(?i)dokumentation\s+online-beteiligung\s+und\s+aufsuchende\s+beteiligung",
    r"(?i)online-beteiligung\s+und\s+aufsuchende\s+formate",
    r"(?i)erstellt\s+im\s+auftrag\s+(der|des)?\s*\w*",
    r"(?i)kokonsult(\s+gmbh)?",
    r"(?i)heidelberg\s+stek\s+2035",
    r"\|\s*25\.06\.-25\.07\.24\s*\|",
    r"\d{2}\.\d{2}\.\d{4}",
]
FOOTER_LINE_PATTERNS = [
    r"(?i)^\s*seite\s+\d+",
    r"(?i)^\s*dokumentation\s*$",
    r"^\s*\d{1,3}\s*$",
]

_inline_re = [re.compile(p) for p in INLINE_PATTERNS]
_footer_re = [re.compile(p) for p in FOOTER_LINE_PATTERNS]


def clean_text(text: str) -> str:
    text = unicodedata.normalize("NFC", text)
    lines = text.split("\n")
    lines = [ln for ln in lines if not any(p.search(ln) for p in _footer_re)]
    text = "\n".join(lines)
    for p in _inline_re:
        text = p.sub(" ", text)
    # rejoin words hyphenated across a line break: "gemes-\nsen" -> "gemessen"
    text = re.sub(r"([a-zäöüß])-\s*\n\s*([a-zäöüß])", r"\1\2", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


## Step 3 — chunk the full document into ~250-word windows

Splits the cleaned document text into consecutive ~`CHUNK_TARGET_TOKENS`-word
windows — the same approach as `chunk_document()` in `stek_pipeline.py`. Each
chunk records its word-offset span (`word_start`-`word_end`) for traceability.
A trailing chunk shorter than `CHUNK_MIN_TOKENS` is merged into the previous
chunk instead of being dropped, so no document text is silently lost.


In [ ]:
def chunk_text(text: str, target_tokens: int, min_tokens: int) -> list[dict]:
    words = text.split()
    chunks = []
    for start in range(0, len(words), target_tokens):
        window = words[start:start + target_tokens]
        chunks.append({
            "text": " ".join(window),
            "word_start": start,
            "word_end": start + len(window) - 1,
        })
    # merge an undersized final chunk into its predecessor
    if len(chunks) >= 2 and len(chunks[-1]["text"].split()) < min_tokens:
        tail = chunks.pop()
        chunks[-1]["text"] += " " + tail["text"]
        chunks[-1]["word_end"] = tail["word_end"]
    return chunks


## Step 4 — topic tagging

A topic counts as matched if any of its keywords appears in the chunk
(word-boundary, case-insensitive). Bigram keywords like `weg_ziel` are
matched as the phrase "weg ziel". `relevance_score` is the number of
distinct keywords matched across all matched topics — the same definition
used in `scraped_data_topics/manifest.json`.

Word-boundary matching is used instead of naive substring matching (which is
what produced `scraped_data_topics`) to avoid false positives such as
`raum` matching inside `Baum`.


In [ ]:
def tag_topics(text: str, topic_keywords: dict[int, list[str]]):
    lower = text.lower()
    matched_topics = []
    matched_keywords = []
    seen = set()
    for topic_id, keywords in sorted(topic_keywords.items()):
        topic_hit = False
        for kw in keywords:
            needle = kw.replace("_", " ").lower()
            if re.search(rf"\b{re.escape(needle)}\b", lower):
                topic_hit = True
                if kw not in seen:
                    seen.add(kw)
                    matched_keywords.append(kw)
        if topic_hit:
            matched_topics.append(topic_id)
    return matched_topics, matched_keywords, len(matched_keywords)


## Step 5 — write one clean, topic-tagged file per chunk

In [ ]:
def slugify(name: str) -> str:
    name = re.sub(r"[^A-Za-z0-9]+", "_", name)
    return re.sub(r"_+", "_", name).strip("_")


def write_chunk_file(out_dir: Path, source_file: str, document_id: str, chunk_id: int,
                      word_start, word_end, matched_topics, matched_keywords, body: str) -> Path:
    header = (
        f"SOURCE_FILE: {source_file}\n"
        f"DOCUMENT_ID: {document_id}\n"
        f"CHUNK_ID: {chunk_id}\n"
        f"WORD_SPAN: {word_start}-{word_end}\n"
        f"MATCHED_TOPICS: {matched_topics}\n"
        f"MATCHED_KEYWORDS: {matched_keywords}\n\n"
    )
    digest = hashlib.md5(f"{document_id}_{chunk_id}".encode("utf-8")).hexdigest()[:10]
    fname = f"{slugify(document_id)}_{chunk_id:03d}_{digest}.txt"
    path = out_dir / fname
    path.write_text(header + body, encoding="utf-8")
    return path


## Step 6 — run the full pipeline over all 12 text files

In [ ]:
txt_files = sorted(TEXT_INPUT_DIR.glob("*.txt"))
print(f"Found {len(txt_files)} text files in {TEXT_INPUT_DIR}")

manifest = []
for txt_path in txt_files:
    source_file = txt_path.name
    document_id = txt_path.stem

    raw = txt_path.read_text(encoding="utf-8")
    cleaned = clean_text(raw)
    if not cleaned:
        print(f"  ! {source_file}: empty after cleaning, skipped")
        continue

    chunks = chunk_text(cleaned, CHUNK_TARGET_TOKENS, CHUNK_MIN_TOKENS)

    for chunk_id, chunk in enumerate(chunks):
        topics, keywords, score = tag_topics(chunk["text"], TOPIC_KEYWORDS)
        out_path = write_chunk_file(
            TEXT_DIR, source_file, document_id, chunk_id,
            chunk["word_start"], chunk["word_end"],
            topics, keywords, chunk["text"],
        )
        manifest.append({
            "source_file": source_file,
            "document_id": document_id,
            "chunk_id": chunk_id,
            "word_span": f"{chunk['word_start']}-{chunk['word_end']}",
            "type": "document",
            "path": str(out_path),
            "relevance_score": score,
            "topics": topics,
        })

    print(f"  ok  {source_file}: {len(cleaned.split())} words -> {len(chunks)} chunks")

MANIFEST_PATH.write_text(json.dumps(manifest, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"\nWrote {len(manifest)} chunks to {TEXT_DIR}")
print(f"Manifest: {MANIFEST_PATH}")


## Step 7 — sanity check

In [ ]:
topic_counts = Counter()
no_topic = 0
for entry in manifest:
    if entry["topics"]:
        topic_counts.update(entry["topics"])
    else:
        no_topic += 1

print("Chunks per document:")
doc_counts = Counter(e["document_id"] for e in manifest)
for doc, n in doc_counts.items():
    print(f"  {n:4d}  {doc}")

print("\nChunks matched per topic (0-4):")
for tid in sorted(TOPIC_KEYWORDS):
    print(f"  Topic {tid}: {topic_counts.get(tid, 0)} chunks")

print(f"\nChunks with zero matched topics: {no_topic} / {len(manifest)}")
print(f"Average relevance_score: {sum(e['relevance_score'] for e in manifest) / max(len(manifest), 1):.1f}")


## Next step (not in this notebook)

Combine `input_data_topics/manifest.json` with `scraped_data_topics/manifest.json`
into a single manifest covering all chunks (12 STEK documents + website HTML +
website PDFs), then run one embedding pass over the union — see the earlier
discussion on why embedding should happen once, after both sources are
combined, rather than separately per source.
